# 📖 Notebook 1 — Leader–Follower Replication: The Basics

Welcome! This is the first notebook in the **replication** lab. We'll answer three simple
questions:

1. **What is replication?** (copying your data to more than one machine)
2. **Why do we bother?** (staying online when a machine dies, and reading faster)
3. **What does "leader–follower" actually look like when you run it?**

By the end of this notebook you will have a real PostgreSQL **primary** and **replica**
running on your laptop in Docker, you will write data to the primary, and you will watch
it *show up on its own* on the replica.

## Learning objectives

- Explain replication in one sentence to a friend.
- Describe the leader–follower (primary–replica) pattern.
- Connect to a Postgres primary **and** a streaming-replication replica with `psycopg2`.
- Observe that writes only work on the primary and that reads work on both.
- Use `pg_stat_replication` to see that the primary knows about its replica.


## 🛠️ Setup

From this lab's folder (`01-foundations/replication/`):

```bash
# 1. start the two-node Postgres cluster + Adminer
docker compose up -d

# 2. install Python deps into a local .venv managed by uv
uv sync
```

**Kernel selection.** In VS Code, open this notebook and pick the `.venv` kernel from the
kernel picker at the top-right of the notebook. If it doesn't appear in the list, reload
the window: `Cmd+Shift+P` → "Reload Window".

**Visualization tool.** Adminer is at <http://localhost:8080>. Log in with:

| Field    | Value                |
|----------|----------------------|
| System   | `PostgreSQL`         |
| Server   | `primary` *(or `replica` if you want to watch the follower)* |
| Username | `demo`               |
| Password | `demo`               |
| Database | `replication_demo`   |

> Tip: Adminer runs inside the Docker network, so from its login screen the server name
> is just `primary` or `replica` — **not** `localhost`.


## 1. What is replication?

> **Replication = keeping the same data on more than one machine, and keeping those
> copies in sync automatically.**

That's the whole idea. Everything else is a detail about *how* the copies stay in sync.

### Why would you do that?

Two big reasons:

1. **Availability.** If the only machine holding your users' data catches fire, your
   product is down until someone restores a backup. With a replica, you can flip traffic
   to the replica and keep serving users.
2. **Read scaling.** Most real apps read far more than they write (think of how often
   you refresh Twitter vs. how often you tweet). If you have 5 replicas, you can
   handle *roughly 5×* the read traffic — each replica has a full copy of the data.

### Leader and follower

There are a lot of replication flavours (we'll meet more later). The simplest and most
common one is **leader–follower**, also called **primary–replica**:

```
                   writes only
          client ──────────────►  ┌──────────┐
                                  │ PRIMARY  │  (the "leader")
                                  └─────┬────┘
                                        │ WAL stream
                                        ▼
                                  ┌──────────┐
          client ◄── reads ──     │ REPLICA  │  (the "follower", read-only)
                                  └──────────┘
```

- The **primary** is the only node that accepts writes.
- The **replica** copies the primary's change log (in Postgres: the **WAL** —
  Write-Ahead Log) and replays it, so it always has the same data a little later.
- Clients can read from either node, but may only write to the primary.

That's exactly what your `docker compose` just started: two Postgres 16 containers, one
configured as the primary (port `5432`) and one as a streaming-replication replica
(port `5433`).


## 2. Connect to both nodes

We'll use a tiny Pydantic model to hold connection info. Pydantic v2 will validate the
types for us, which is nice when you have two connection strings that look almost
identical.


In [1]:
import psycopg2
from pydantic import BaseModel, Field


class PgNode(BaseModel):
    name: str
    host: str = "localhost"
    port: int = Field(ge=1, le=65535)
    dbname: str = "replication_demo"
    user: str = "demo"
    password: str = "demo"

    def connect(self):
        return psycopg2.connect(
            host=self.host, port=self.port,
            dbname=self.dbname, user=self.user, password=self.password,
        )


PRIMARY = PgNode(name="primary", port=5432)
REPLICA = PgNode(name="replica", port=5433)

PRIMARY, REPLICA


(PgNode(name='primary', host='localhost', port=5432, dbname='replication_demo', user='demo', password='demo'),
 PgNode(name='replica', host='localhost', port=5433, dbname='replication_demo', user='demo', password='demo'))

In [2]:
def ping(node: PgNode) -> str:
    with node.connect() as conn, conn.cursor() as cur:
        cur.execute("SELECT pg_is_in_recovery();")
        in_recovery, = cur.fetchone()
        role = "replica (read-only, in recovery)" if in_recovery else "primary (read-write)"
        return f"{node.name} @ {node.host}:{node.port} -> {role}"


print(ping(PRIMARY))
print(ping(REPLICA))


primary @ localhost:5432 -> primary (read-write)
replica @ localhost:5433 -> replica (read-only, in recovery)


`pg_is_in_recovery()` is Postgres' own way of answering *"am I a replica?"*. A node
returns `true` if it is replaying WAL from somewhere else — i.e. it's a follower. The
primary returns `false`.

## 3. Write to the primary, read from both

The init script created a `users` table with Alice, Bob, and Charlie. Let's add a new
user on the primary and then read from both nodes and watch the row appear on the
replica.


In [3]:
def all_users(node: PgNode):
    with node.connect() as conn, conn.cursor() as cur:
        cur.execute("SELECT id, username, email FROM users ORDER BY id;")
        return cur.fetchall()


# Add a user on the PRIMARY
with PRIMARY.connect() as conn, conn.cursor() as cur:
    cur.execute(
        "INSERT INTO users (username, email) VALUES (%s, %s) ON CONFLICT (username) DO NOTHING;",
        ("diana", "diana@example.com"),
    )
    conn.commit()

print("PRIMARY users:", all_users(PRIMARY))
print("REPLICA users:", all_users(REPLICA))


PRIMARY users: [(1, 'alice', 'alice@example.com'), (2, 'bob', 'bob@example.com'), (3, 'charlie', 'charlie@example.com'), (4, 'dave', 'dave@example.com'), (5, 'diana', 'diana@example.com')]
REPLICA users: [(1, 'alice', 'alice@example.com'), (2, 'bob', 'bob@example.com'), (3, 'charlie', 'charlie@example.com'), (4, 'dave', 'dave@example.com'), (5, 'diana', 'diana@example.com')]


You should see the *same rows on both nodes*, including `diana`. The replica did not
run any SQL we gave it — it saw the INSERT on the primary's WAL and replayed it.

## 4. What happens if we try to write to the replica?


In [4]:
try:
    with REPLICA.connect() as conn, conn.cursor() as cur:
        cur.execute(
            "INSERT INTO users (username, email) VALUES (%s, %s);",
            ("eve", "eve@example.com"),
        )
        conn.commit()
except psycopg2.errors.ReadOnlySqlTransaction as e:
    print("Write to replica was refused, as expected:")
    print("  ", e)


Write to replica was refused, as expected:
   cannot execute INSERT in a read-only transaction



Good — the replica is **read-only** on purpose. If you could write to it, the two
nodes could easily drift out of sync and you'd have two different truths. Leader–
follower deliberately funnels all writes through one node to avoid that.

## 5. Look at replication from the primary's point of view

Postgres exposes a system view called `pg_stat_replication` that shows every connected
replica, how far behind it is, and whether it's synchronous or asynchronous.


In [5]:
with PRIMARY.connect() as conn, conn.cursor() as cur:
    cur.execute('''
        SELECT application_name, client_addr, state, sync_state,
               pg_wal_lsn_diff(sent_lsn, replay_lsn) AS lag_bytes
        FROM pg_stat_replication;
    ''')
    cols = [d.name for d in cur.description]
    for row in cur.fetchall():
        print(dict(zip(cols, row)))


{'application_name': 'walreceiver', 'client_addr': '172.27.0.4', 'state': 'streaming', 'sync_state': 'async', 'lag_bytes': Decimal('0')}


A few things to notice:

- **`state = streaming`** — the replica has caught up and is tailing the WAL live.
- **`sync_state = async`** — this replica is asynchronous. The primary does **not**
  wait for it before acknowledging a client's write. That's fast, but it means the
  replica might be a little behind. Notebook 2 will explore that tradeoff.
- **`lag_bytes`** — how many WAL bytes the primary has sent that the replica has not
  yet replayed. On an idle system this is `0`.

## 6. Recap

- Replication = same data, multiple machines, kept in sync automatically.
- **Leader–follower (primary–replica)** is the simplest flavour: writes go to one node,
  reads can go to any node.
- The replica is really just a Postgres instance in *recovery mode* replaying the
  primary's WAL.
- Writes to the replica are refused at the database level — the system stops you from
  accidentally splitting the truth.
- Adminer (<http://localhost:8080>) lets you connect to `primary` or `replica` and see
  the rows yourself.

### What's next

Notebook 2 zooms in on the **synchronous vs asynchronous** tradeoff: if the primary
doesn't wait for the replica, clients can accidentally read **stale** data. We'll
simulate that lag in pure Python so you can feel it.
